# AAND Stage 2: Normality Distillation

In Stage 1, we made the Teacher more sensitive to anomalies. In Stage 2, we train the Student to copy the Teacher.

However, some normal textures (like fine-grained wire rope details) are harder to reconstruct than flat backgrounds. If the Student struggles to reconstruct them, they might trigger false positive anomalies.

To solve this, AAND introduces the **Hard Knowledge Distillation (HKD) Loss**. This loss explicitly forces the Student to focus on the hardest-to-reconstruct normal patches.

In [ ]:
import os
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torchvision.transforms as transforms
from glob import glob

from models import AdvancedDINOv2Teacher, CustomStudentCNN
from train_aand import HardKnowledgeDistillationLoss

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Setup Models
We load the Advanced Teacher and freeze it completely. We also initialize a fresh Student CNN.

In [ ]:
# Load Frozen Advanced Teacher
teacher = AdvancedDINOv2Teacher(size='large').to(device)
teacher.freeze_raa()
teacher.eval()

# Load Learnable Student
student = CustomStudentCNN(out_channels=1024).to(device)
student.train()

## 2. Hard Knowledge Distillation (HKD) Analysis
Let's see what happens when we pass a normal image through both networks before any training has occurred. The Student's outputs will be essentially random, meaning the reconstruction error will be high everywhere. 

We will calculate the per-patch cosine distance and see which patches the HKD loss selects as "hard".

In [ ]:
# Load a sample normal image
images = sorted(glob('dataset/train/images/*.*'))
if len(images) == 0:
    img = Image.fromarray(np.random.randint(50, 200, (518, 518, 3), dtype=np.uint8))
else:
    img = Image.open(images[0]).convert('RGB').resize((518, 518))

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
input_tensor = transform(img).unsqueeze(0).to(device)

# Forward pass
with torch.no_grad():
    # Teacher (Stage 2: gt_mask is None)
    t_feat, _, _ = teacher(input_tensor, gt_mask=None)
    t_feat = F.normalize(t_feat, dim=1, p=2)
    
    # Student
    s_feat = student(input_tensor)
    s_feat = F.normalize(s_feat, dim=1, p=2)

# Calculate per-patch Cosine Distance (1 - Cosine Similarity)
cos_sim = (s_feat * t_feat).sum(dim=1)  # (1, 37, 37)
patch_errors = 1 - cos_sim

print("Mean Patch Error (Standard KD Loss):", patch_errors.mean().item())

Now let's identify the Top-10 Hardest Patches (K_h = 10).

In [ ]:
K_h = 10
errors_flat = patch_errors.flatten() # (37 * 37 = 1369)
topk_errors, topk_indices = torch.topk(errors_flat, K_h)

print(f"Top {K_h} Errors (HKD Loss Component):", topk_errors.mean().item())

# Create a mask showing where the hard patches are
hard_mask = torch.zeros_like(errors_flat)
hard_mask[topk_indices] = 1
hard_mask = hard_mask.view(37, 37)

# Visualize
patch_errors_vis = patch_errors.squeeze().cpu().numpy()
hard_mask_vis = F.interpolate(hard_mask.unsqueeze(0).unsqueeze(0), size=(518, 518), mode='nearest').squeeze().cpu().numpy()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(img)
axes[0].set_title("Input Normal Image")
axes[0].axis('off')

im = axes[1].imshow(patch_errors_vis, cmap='hot')
axes[1].set_title("All Patch Errors (1 - Cosine Sim)")
axes[1].axis('off')
plt.colorbar(im, ax=axes[1])

axes[2].imshow(img)
axes[2].imshow(hard_mask_vis, cmap='spring', alpha=0.5)
axes[2].set_title(f"Top-{K_h} Hardest Patches Highlighted")
axes[2].axis('off')

plt.tight_layout()
plt.show()

### Why this works:
During training, the loss function is calculated as:
`Total Loss = (Mean of ALL patches) + lambda * (Mean of TOP-K hardest patches)`

Because gradients flow strongly from the hardest patches back into the student network, the student is forced to allocate its capacity to learning the complex textures present in those hard patches, rather than just learning the easy, smooth background.

Once training is complete, the student will have accurately mapped the normal distribution of the Advanced Teacher, and any deviations during inference will be flagged as anomalies!